In [1]:
# Step 1: create the repository's own virtual environment (.venv) and install everything in it.
# Nothing is installed into the Python that runs this notebook.
import hashlib
import json
import os
import subprocess
import sys
import venv
from pathlib import Path

TORCH_PACKAGES = ["torch"]
TORCH_INDEX_URL = "https://download.pytorch.org/whl/cu126"
PACKAGES = [
    "biopython",
    "numpy",
    "pandas",
    "pennylane",
    "psutil",
    "rdkit",
    "scikit-learn",
]

ROOT = Path.cwd().resolve()
if not (ROOT / "run_experiment.py").exists():
    raise RuntimeError(
        f"Open this notebook from the repository folder (run_experiment.py not found in {ROOT})"
    )
if sys.version_info < (3, 10):
    raise RuntimeError("Python 3.10 or newer is needed (pennylane)")

VENV_DIR = ROOT / ".venv"
PY = VENV_DIR / ("Scripts/python.exe" if os.name == "nt" else "bin/python")


def stream(command):
    process = subprocess.Popen(
        [str(c) for c in command],
        cwd=ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end="")
    code = process.wait()
    if code != 0:
        raise RuntimeError(f"Command failed with exit code {code}: {command}")


if not PY.exists():
    print(f"Creating virtual environment in {VENV_DIR}")
    venv.EnvBuilder(with_pip=True).create(VENV_DIR)

marker = VENV_DIR / ".packages_installed"
stamp = hashlib.sha256(
    json.dumps([TORCH_PACKAGES, TORCH_INDEX_URL, PACKAGES]).encode()
).hexdigest()
if not marker.exists() or marker.read_text() != stamp:
    stream([PY, "-m", "pip", "install", "--upgrade", "pip"])
    stream([PY, "-m", "pip", "install", *TORCH_PACKAGES, "--index-url", TORCH_INDEX_URL])
    stream([PY, "-m", "pip", "install", *PACKAGES])
    marker.write_text(stamp)
print("Virtual environment ready:", PY)


Creating virtual environment in C:\Users\vm\Downloads\queen-dta-deeph-protocol (1)\queen-dta-deeph-protocol\.venv
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   - -------------------------------------- 0.1/1.8 MB 3.2 MB/s eta 0:00:01
   - -------------------------------------- 0.1/1.8 MB 1.3 MB/s eta 0:00:02
   - -------------------------------------- 0.1/1.8 MB 1.3 MB/s eta 0:00:02
   ----- ---------------------------------- 0.2/1.8 MB 1.3 MB/s eta 0:00:02
   ------- -------------------------------- 0.4/1.8 MB 1.7 MB/s eta 0:00:01
   --------- ------------------------------ 0.4/1.8 MB 1.8 MB/s eta 0:00:01
   --------- ------------------------------ 0.4/1.8 MB 1.8 MB/s eta 0:00:01
   --------- ------------------------------ 0.4/1.8 MB 1.4 MB/s eta 0:00:02
   --------- ------------------------------ 0.5/1.8 MB 1.3 MB/s eta 0:0

In [2]:
# Step 2: check that the virtual environment sees the GPU and the data is intact.
stream([PY, "-c", (
    "import torch;"
    "assert torch.cuda.is_available(), 'No CUDA GPU: update the NVIDIA driver or change TORCH_INDEX_URL';"
    "p = torch.cuda.get_device_properties(0);"
    "print('GPU:', p.name, round(p.total_memory / 2**30, 1), 'GB | torch', torch.__version__)"
)])
stream([PY, "-c", (
    "from queen_dta.data import load_dataset, check_ids;"
    "[print(d, len(load_dataset(d)[0]), 'pairs, folds OK') for d in ('davis', 'kiba')]"
)])


GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU 6.0 GB | torch 2.14.1+cu126
davis 30056 pairs, folds OK
kiba 118254 pairs, folds OK


In [3]:
# Step 3: train QuEEN-DTA with the DeepH-DTA protocol (200 epochs per run, 5 runs per dataset).
# Finished runs are saved in results/ and skipped when you run this cell again.
# For a quick test use DATASETS = ["davis"] and FOLDS = [0].
DATASETS = ["davis", "kiba"]
FOLDS = [0, 1, 2, 3, 4]

stream([PY, "-u", "run_experiment.py", "--datasets", *DATASETS, "--folds", *[str(f) for f in FOLDS]])


GPU memory free: 5.0 of 6.0 GB
PyTorch GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU | 6.0 GB | CUDA 12.6
Keep-awake on: the computer will not sleep and the screen will stay on until the run ends
davis: 30,056 pairs, building features ...
davis | validation fold 0 | train 20,036 | validation 5,010 | test 5,010
davis fold 0 epoch 10 | val MSE 0.3833 | best 0.3789 (epoch 9)
davis fold 0 epoch 20 | val MSE 0.3123 | best 0.3061 (epoch 19)
davis fold 0 epoch 30 | val MSE 0.2779 | best 0.2775 (epoch 27)
davis fold 0 epoch 40 | val MSE 0.2635 | best 0.2537 (epoch 36)
davis fold 0 epoch 50 | val MSE 0.2414 | best 0.2414 (epoch 50)
davis fold 0 epoch 60 | val MSE 0.2401 | best 0.2401 (epoch 60)
davis fold 0 epoch 70 | val MSE 0.2302 | best 0.2302 (epoch 70)
davis fold 0 epoch 80 | val MSE 0.2357 | best 0.2302 (epoch 70)
davis fold 0 epoch 90 | val MSE 0.2330 | best 0.2300 (epoch 89)
davis fold 0 epoch 100 | val MSE 0.2387 | best 0.2282 (epoch 99)
davis fold 0 epoch 110 | val MSE 0.2244 | best 0.2

In [4]:
# Step 4: results (also saved in results/*.csv)
stream([PY, "report.py"])


Per fold (validation fold = run):
dataset  val_fold  seed  best_epoch  val_mse  test_mse  test_ci  test_r2m  test_r2
  Davis         0     0         181   0.2144    0.2255   0.8890    0.7087   0.7186
  Davis         1     1         196   0.2047    0.2314   0.8913    0.6501   0.7112
  Davis         2     2         176   0.2009    0.2400   0.8896    0.6367   0.7005
  Davis         3     3         189   0.2000    0.2353   0.8847    0.6652   0.7065
  Davis         4     4         173   0.2049    0.2351   0.8846    0.6538   0.7066
   KIBA         0     0         130   0.1617    0.1748   0.8767    0.6743   0.7428
   KIBA         1     1         200   0.1488    0.1521   0.8832    0.7236   0.7763
   KIBA         2     2         194   0.1516    0.1488   0.8846    0.7338   0.7811
   KIBA         3     3         197   0.1628    0.1534   0.8843    0.7188   0.7743
   KIBA         4     4         192   0.1465    0.1466   0.8848    0.7444   0.7843

QuEEN-DTA, mean (std) over the folds, fixed test set